<a href="https://colab.research.google.com/github/Fisev/PZP-Project/blob/Fisev/Project.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Effective single threaded on CPU

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [1]:
!wget https://raw.githubusercontent.com/Fisev/PZP-Project/refs/heads/main/data.txt -O data.txt
!wget https://raw.githubusercontent.com/Fisev/PZP-Project/refs/heads/main/stop_words.txt -O stop_words.txt

with open("data.txt", 'r') as file:
    whole_text = file.read()

with open("stop_words.txt", 'r') as file:
    stop_words = file.read()

--2024-12-03 14:33:50--  https://raw.githubusercontent.com/Fisev/PZP-Project/refs/heads/main/data.txt
Resolving raw.githubusercontent.com (raw.githubusercontent.com)... 185.199.108.133, 185.199.109.133, 185.199.110.133, ...
Connecting to raw.githubusercontent.com (raw.githubusercontent.com)|185.199.108.133|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 1257260 (1.2M) [text/plain]
Saving to: ‘data.txt’

data.txt            100%[===================>]   1.20M  --.-KB/s    in 0.05s   

2024-12-03 14:33:50 (23.8 MB/s) - ‘data.txt’ saved [1257260/1257260]

--2024-12-03 14:33:50--  https://raw.githubusercontent.com/Fisev/PZP-Project/refs/heads/main/stop_words.txt
Resolving raw.githubusercontent.com (raw.githubusercontent.com)... 185.199.110.133, 185.199.109.133, 185.199.111.133, ...
Connecting to raw.githubusercontent.com (raw.githubusercontent.com)|185.199.110.133|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 100 [text/plain]
Saving to: 

In [4]:
!pip install pycuda

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.7/1.7 MB 26.5 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 89.8/89.8 kB 8.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 78.6/78.6 kB 7.7 MB/s eta 0:00:00
  Created wheel for pycuda: filename=pycuda-2024.1.2-cp310-cp310-linux_x86_64.whl size=660545 sha256=79a467cd911cb0491b5d35488e510d1ec1c908a9c4380d71172cca24e19c66e4
  Stored in directory: /root/.cache/pip/wheels/70/63/40/4bf006182f942d3516b71bb2ff3b57ccbdb8b2c0ee81882b6e
Successfully built pycuda


Single threaded on CPU


In [10]:
import re
from collections import Counter
from collections import defaultdict
import time

start_time = time.time()

words = re.findall(r'\b[a-zA-Z]+(?:-[a-zA-Z]+)?\b', whole_text.lower())

splitted_stop_words = stop_words.split()

cleaned_words = [word for word in words if (word not in splitted_stop_words) and (len(word) <= 8 and len(word) >=4)]

word_counts = Counter(cleaned_words)
most_common_word, occurrences_common_word = word_counts.most_common(1)[0]

end_time = time.time()
print(f"time elapsed: {end_time - start_time}")
print(f"most common word: '{most_common_word}' with occurrences {occurrences_common_word}")

least_common_word, occurrences_least_common_word = word_counts.most_common()[-1]
print(f"The least frequent word is '{least_common_word}' with {occurrences_least_common_word} occurrences.")

print(f"Total number of words: {len(cleaned_words)}")


time elapsed: 0.17300820350646973
most common word: 'that' with occurrences 3100
The least frequent word is 'includes' with 1 occurrences.
Total number of words: 107928


Multithreaded on CPU

In [36]:
import threading
import multiprocessing
import time
from collections import Counter

def text_procesing(text_chunk, target_words):
    splitted_text_chunk = re.findall(r'\b[a-zA-Z]+(?:-[a-zA-Z]+)?\b', text_chunk.lower())

    filtered_words = []
    word_counter = Counter()
    i = 0
    for word in splitted_text_chunk:
        if len(word) <= 8 and len(word) >= 4:
            if word not in target_words:
                word_counter[word] += 1
                i += 1
                filtered_words.append(word)

    return (word_counter, filtered_words)

start_time = time.time()

splitted_stop_words = stop_words.split()

num_chunks = cpu_core_count = multiprocessing.cpu_count()
chunk_size = len(whole_text) // num_chunks

print(f"Number of CPU cores available: {cpu_core_count}")

chunks = [""] * num_chunks
for i in range(0, num_chunks):
    j = 0
    if i != 0:
        while whole_text[i + j + chunk_size] != " ":
            j += 1

    chunks[i] = whole_text[i * chunk_size + j:i * chunk_size + chunk_size + j]

with multiprocessing.Pool(processes=multiprocessing.cpu_count()) as pool:
    results = pool.starmap(text_procesing, [(chunk, splitted_stop_words) for chunk in chunks])

combined_counts = Counter()
filtered_words = []
for result in results:
    combined_counts += result[0]
    filtered_words.extend(result[1])

total_word_count = len(filtered_words)
end_time = time.time()

most_common_word_mul, most_common_freq_mul = combined_counts.most_common(1)[0]
least_common_word_mul, least_common_freq_mul = combined_counts.most_common()[-1]

print(f"Number of words: {total_word_count}")
print(f"The most frequent word is: '{most_common_word_mul}' with {most_common_freq_mul} occurrences")
print(f"The least frequent word: '{least_common_freq_mul}' with {least_common_freq_mul} occurrences")
print(f"time elapsed: {end_time - start_time}")

Number of CPU cores available: 2
Number of words: 107927
The most frequent word is: 'that' with 3100 occurrences
The least frequent word: '1' with 1 occurrences
time elapsed: 0.3100316524505615


GPU, easy way

In [37]:
import numpy as np
import re
from pycuda.compiler import SourceModule
import pycuda.autoinit
from pycuda import gpuarray
import pycuda.driver as cuda
import time

cuda.init()
dev = cuda.Device(0)
ctx = dev.make_context()

TextAnalysis='''

__global__ void text_analysis(int *encoded_words, int *encoded_stop_words, const int *words_count, int* encoded_words_out, int *words_histogram)
{
    int N = *words_count;
    __shared__ int stop_words[6];
    int idx = blockIdx.x * blockDim.x + threadIdx.x;

    if (idx < 6){
        stop_words[idx] = encoded_stop_words[idx];
    }
    __syncthreads();

    if (idx < N) {
        size_t i = 0;
        for (; i < 6; ++i){
            if (encoded_words[idx] == stop_words[i]){
                break;
            }
            if (i == 5){
                atomicAdd(&words_histogram[encoded_words[idx]], 1);
            }
        }
    }

}
'''

start_time = time.time()

# (?:-[a-zA-Z]+) is optional becouse of ? at the end, ?: makes it Capturing - it will not create new group
words = re.findall(r'\b[a-zA-Z]+(?:-[a-zA-Z]+)?\b', whole_text.lower())
words = [word for word in words if len(word) <= 8 and len(word) >=4]

splitted_stop_words = stop_words.lower().split()
splitted_stop_words = [word for word in splitted_stop_words if len(word) <= 8 and len(word) >=4]

unique_words = list(set(words))
word_to_num = {word: idx for idx, word in enumerate(unique_words)}
num_to_word = {idx: word for word, idx in word_to_num.items()}

encoded_words = [word_to_num[word] for word in words]
encoded_words = np.array(encoded_words, dtype=np.int32)
encoded_stop_words = [word_to_num[word] for word in splitted_stop_words]
encoded_stop_words = np.array(encoded_stop_words, dtype=np.int32)

BLOCK_SIZE = 256
gridDim = len(encoded_words) // BLOCK_SIZE + 1

text_analysis_mod = SourceModule(TextAnalysis)
Text_analysis_kernel = text_analysis_mod.get_function('text_analysis')

encoded_words_out = np.zeros(encoded_words.nbytes).astype(np.int32)
words_histogram = np.zeros(encoded_words.nbytes).astype(np.int32)
word_count = np.zeros(1).astype(np.int32)
word_count[0] = len(encoded_words)

encoded_words_in_gpu = cuda.mem_alloc(encoded_words.nbytes)
encoded_words_out_gpu = cuda.mem_alloc(encoded_words_out.nbytes)
encoded_stop_words_gpu = cuda.mem_alloc(encoded_stop_words.nbytes)
words_histogram_gpu = cuda.mem_alloc(words_histogram.nbytes)
word_count_gpu = cuda.mem_alloc(word_count.nbytes)

cuda.memcpy_htod(encoded_words_in_gpu, encoded_words)
cuda.memcpy_htod(encoded_stop_words_gpu, encoded_stop_words)
cuda.memcpy_htod(word_count_gpu, word_count)
cuda.memcpy_htod(words_histogram_gpu, words_histogram)

Text_analysis_kernel(encoded_words_in_gpu, encoded_stop_words_gpu, word_count_gpu, encoded_words_out_gpu, words_histogram_gpu, block=(BLOCK_SIZE,1,1), grid=(gridDim,1,1))
ctx.synchronize()

cuda.memcpy_dtoh(encoded_words_out, encoded_words_out_gpu)
cuda.memcpy_dtoh(words_histogram, words_histogram_gpu)

most_common_word_encoded = np.argmax(words_histogram)
most_common_word__freq = np.max(words_histogram)
most_common_word = num_to_word[most_common_word_encoded]

masked_histogram = np.where(words_histogram == 0, np.inf, words_histogram)
least_common_word_encoded = np.argmin(masked_histogram)
least_common_word__freq = np.min(masked_histogram)
least_common_word = num_to_word[least_common_word_encoded]

end_time = time.time()
print(f"Time elapsed: {end_time - start_time}")
print(f"Total number of words: {len(words)}")
print(f"The most common word is: '{most_common_word}' frequency: {most_common_word__freq}")
print(f"The least common word is: '{least_common_word}' frequency: {int(least_common_word__freq)}")

encoded_words_in_gpu.free()
encoded_stop_words_gpu.free()
word_count_gpu.free()
ctx.pop()

Time elapsed: 0.17627573013305664
Total number of words: 108064
The most common word is: 'that' frequency: 3100
The least common word is: 'flexion' frequency: 1


Single threaded on CPU trying to be more effective

In [11]:
import re
from collections import Counter
from collections import defaultdict
import time

start_time = time.time()
words = re.findall(r'\b[a-zA-Z]+(?:-[a-zA-Z]+)?\b', whole_text.lower())

splitted_stop_words = stop_words.split()

filtered_words = [""] * 120000
word_counts = defaultdict(int)
max_key, max_value = "", 0
min_key, min_value = "", 0
i = 0
for word in words:
    if len(word) <= 8 and len(word) >= 4:
        if word not in splitted_stop_words:
            word_counts[word] += 1
            i += 1
            filtered_words[i] = word
            value = word_counts[word]

            if word_counts[word] > max_value:
                max_key, max_value = word, value

min_key, min_value = min(word_counts.items(), key=lambda item: item[1])
end_time = time.time()

print(f"Number of words: {i}")
print(f"The most frequent word is: '{max_key}' with {max_value} occurrences")
print(f"The least frequent word: '{min_key}' with {min_value} occurrences")
print(f"time elapsed: {end_time - start_time}")

Number of words: 107928
The most frequent word is: 'that' with 3100 occurrences
The least frequent word: 'january' with 1 occurrences
time elapsed: 0.21302485466003418


GPU, not finished, maybe philosophicly more correct

In [ ]:
import numpy as np
from pycuda.compiler import SourceModule
import pycuda.autoinit
from pycuda import gpuarray
import pycuda.driver as cuda
import time

cuda.init()
dev = cuda.Device(0)
ctx = dev.make_context()

TextAnalysis='''

__global__ void text_analysis(int *byte_array, const int *offsets, int* num_of_offsets, int *data_out, int *max_out)
{
    int N = *num_of_offsets;
    int idx = blockIdx.x * blockDim.x + threadIdx.x;
    int* chunkStart = NULL;
    int chunkLen =  0;

    if (idx < N) {
        if(idx == 0) {
            chunkStart = byte_array;
            chunkLen = offsets[idx];
        }
        else{
            chunkStart = byte_array + offsets[idx-1] + 1;
            chunkLen = offsets[idx] - offsets[idx - 1];
        }

        //char** words = malloc(len * sizeof(char*) + len/3);
        //int *word_count = 0;

        int data_out_index = 0;
        int start = 0;
        for (size_t i = 0; i < chunkLen; i++) {
            if (!((chunkStart[i] >= 'A' && chunkStart[i] <= 'Z') || (chunkStart[i] >= 'a' && chunkStart[i] <= 'z'))) {
                 if (i > start) {
                     int word_len = i - start;

                     for(size_t j = 0; j < word_len; j++){
                         if (idx == 0){
                            data_out[data_out_index] = chunkStart[start + j];
                         }
                         else{
                              data_out[data_out_index + offsets[idx - 1]] = chunkStart[start + j];
                         }
                         data_out_index++;
                     }
                     data_out[data_out_index] = 32;
                     data_out_index++;

                     //max_out[0] = 10;
/*
                   words[*word_count] = malloc((word_len + 1) * sizeof(char));
                     strncpy(words[*word_count], &str[start], word_len);
                     words[*word_count][word_len] = '\0';
                     (*word_count)++;  */
                 }
            }
                 if (i + 1 < chunkLen) {
                     start = i + 1;
}
        }
    }

}
'''
byte_array_text = np.array(list(whole_text.encode('utf-8')), dtype=np.uint32)
byte_array_stop_words = np.array(list(stop_words.encode('utf-8')), dtype=np.uint32)

print(str(byte_array_text))

# [print(chr(temp)) for temp in byte_array_text]

num_of_chunks = len(byte_array_text) // 6000


chunk_size = len(whole_text) // num_of_chunks
BLOCK_SIZE = 256
gridDim = (num_of_chunks + BLOCK_SIZE - 1) // BLOCK_SIZE
print(f"gridDim: {gridDim}")

offsets = np.empty(num_of_chunks + 1, dtype=np.int32)
for i in range(1, num_of_chunks):
    j = 0
    while byte_array_text[i * chunk_size + j] != 32:
        j += 1
        # print(j)
    offsets[i-1] = i * chunk_size + j

offsets[num_of_chunks] = len(byte_array_text)

text_analysis_mod = SourceModule(TextAnalysis)
Text_analysis_kernel = text_analysis_mod.get_function('text_analysis')

statistics_out = np.zeros(100).astype(np.int32)
chunk_number = np.zeros(1).astype(np.int32)
chunk_number[0] = num_of_chunks
data_out = np.zeros(byte_array_text.nbytes * 2).astype(np.int32)

input_data_gpu = cuda.mem_alloc(byte_array_text.nbytes)
offsets_gpu = cuda.mem_alloc(offsets.nbytes)
chunk_number_gpu = cuda.mem_alloc(chunk_number.nbytes)

data_out_gpu = cuda.mem_alloc(data_out.nbytes)
statistics_out_gpu = cuda.mem_alloc(statistics_out.nbytes)

cuda.memcpy_htod(input_data_gpu, byte_array_text)
cuda.memcpy_htod(offsets_gpu, offsets)
cuda.memcpy_htod(chunk_number_gpu, chunk_number)

Text_analysis_kernel(input_data_gpu, offsets_gpu, chunk_number_gpu, data_out_gpu, statistics_out_gpu, block=(BLOCK_SIZE,1,1), grid=(gridDim,1,1))
ctx.synchronize()

cuda.memcpy_dtoh(data_out, data_out_gpu)
cuda.memcpy_dtoh(statistics_out, statistics_out_gpu)
cuda.memcpy_dtoh(statistics_out, statistics_out_gpu)

print(f"\n\n\n text:")
# [print(chr(temp)) for temp in byte_array_text]
print(data_out)
print(f"\n\n\n max: {statistics_out}")

input_data_gpu.free()
data_out_gpu.free()
ctx.pop()

LogicError: cuCtxCreate failed: an illegal memory access was encountered

GPU easy version